# Netflix SWOT - Monthly update (inference-only)

This notebook runs every month. It does not train anything: it loads the
classifier I trained in the main notebook, applies it to the latest reviews,
rebuilds the topics and the SWOT matrix, and writes data.json for the
dashboard.

Expected attached inputs:
1. `ashishkumarak/netflix-reviews-playstore-daily-updated` (auto-fetched latest via kagglehub anyway)
2. `anastasiapolitidou/netflix-swot-model` — a Kaggle Datasetcreated from the
   full notebook's output folder `netflix-swot-model` (the trained classifier).



## 0. Setup


In [1]:
import os
os.environ["CUDA_VISIBLE_DEVICES"] = "0,1"
os.environ["PYTORCH_CUDA_ALLOC_CONF"] = "expandable_segments:True"

import re, gc, json, glob
import numpy as np
import pandas as pd
import torch

SEED = 42
np.random.seed(SEED)
torch.manual_seed(SEED)

SWOT_LABELS = ["Strength", "Weakness", "Opportunity", "Threat"]
UNCLASSIFIED = "Unclassified"

print("CUDA available:", torch.cuda.is_available(), "| device count:", torch.cuda.device_count())

CUDA available: True | device count: 2


In [2]:
!pip install -q langdetect emoji rapidfuzz bitsandbytes

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 981.5/981.5 kB 14.5 MB/s eta 0:00:00
  Preparing metadata (setup.py) ... done
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.1/3.1 MB 59.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 40.9/40.9 MB 45.1 MB/s eta 0:00:00


## 1. Inference parameters (from the last full training run)


In [3]:
# The training notebook tunes these and saves them next to the model as
# inference_params.json. This run only applies them. The constants below are a
# fallback for the case where the model dataset predates that file.

THRESHOLDS  = {"Strength": 0.55, "Weakness": 0.30, "Opportunity": 0.45, "Threat": 0.20}
ALPHA       = 0.0
PRIOR_RATIO = {"Strength": 0.88, "Weakness": 0.95, "Opportunity": 1.22, "Threat": 0.81}
MACRO_F1    = 0.80   # gold-test macro-F1 of the deployed model (for the dashboard KPI)

_params_hits = glob.glob("/kaggle/input/**/inference_params.json", recursive=True)
if _params_hits:
    with open(_params_hits[0]) as f:
        _p = json.load(f)
    THRESHOLDS  = _p.get("thresholds", THRESHOLDS)
    ALPHA       = _p.get("alpha", ALPHA)
    PRIOR_RATIO = _p.get("prior_ratio", PRIOR_RATIO)
    MACRO_F1    = _p.get("macro_f1", MACRO_F1)
    print("Loaded inference_params.json:", _params_hits[0])
else:
    print("WARNING: no inference_params.json found next to the model; "
          "using the fallback constants above. Check they match the last "
          "training run before trusting the dashboard KPIs.")

thr_vec = np.array([THRESHOLDS[l] for l in SWOT_LABELS])
prior_ratio = np.array([PRIOR_RATIO[l] for l in SWOT_LABELS])
print("thresholds:", THRESHOLDS, "| alpha:", ALPHA)

thresholds: {'Strength': 0.55, 'Weakness': 0.3, 'Opportunity': 0.45, 'Threat': 0.2} | alpha: 0.0


## 2. Latest reviews (kagglehub always fetches the newest version)


In [4]:
import kagglehub

_ds = kagglehub.dataset_download("ashishkumarak/netflix-reviews-playstore-daily-updated")
_csvs = glob.glob(os.path.join(_ds, "*.csv"))
REVIEWS_PATH = _csvs[0]
print("Dataset dir:", _ds)
print("Using file :", REVIEWS_PATH)

df_raw = pd.read_csv(REVIEWS_PATH)
df_raw = df_raw[["content", "score", "at"]].rename(
    columns={"content": "text", "score": "rating", "at": "date"})
df_raw["date"] = pd.to_datetime(df_raw["date"], errors="coerce")
df_raw["rating"] = pd.to_numeric(df_raw["rating"], errors="coerce")
df_raw = df_raw[df_raw["date"] >= "2020-01-01"].reset_index(drop=True)
print("Reviews from 2020 onwards:", len(df_raw))
print("Most recent review date  :", df_raw["date"].max())

Dataset dir: /kaggle/input/datasets/ashishkumarak/netflix-reviews-playstore-daily-updated
Using file : /kaggle/input/datasets/ashishkumarak/netflix-reviews-playstore-daily-updated/netflix_reviews.csv
Reviews from 2020 onwards: 134071
Most recent review date  : 2026-08-12 08:40:47


## 3. Clean the text 
Same as the main notebook


In [5]:
from langdetect import detect, DetectorFactory
from langdetect.lang_detect_exception import LangDetectException
import emoji
from rapidfuzz import process, fuzz

DetectorFactory.seed = SEED

LOW_INFO = {"good", "bad", "nice", "ok", "wow", "love it", "hate it",
            "very good", "very bad", "awesome", "terrible"}
SPAM_PATTERNS = [r"earn money", r"work from home", r"visit my channel",
                 r"subscribe to", r"check out my", r"promo code",
                 r"free followers", r"telegram", r"whatsapp"]

def detect_lang_safe(text):
    try:
        return detect(text)
    except LangDetectException:
        return "unknown"

def clean_text(text):
    text = str(text).strip().lower()
    text = emoji.replace_emoji(text, replace="")
    text = re.sub(r"http\S+|www\S+", " ", text)
    text = re.sub(r"<.*?>", " ", text)
    text = re.sub(r"[^a-zA-Z0-9\s\.\,\!\?\-\'\"]", " ", text)
    text = re.sub(r"\s+", " ", text).strip()
    return text

def is_low_quality(text):
    if not isinstance(text, str):
        return True
    txt = text.strip()
    if len(txt) < 20 or len(txt.split()) < 4:
        return True
    if txt.lower() in LOW_INFO:
        return True
    if re.search(r"(.)\1{5,}", txt):
        return True
    return False

def is_spam_like(text):
    txt = text.lower()
    return any(re.search(p, txt) for p in SPAM_PATTERNS)

def remove_near_duplicates(frame, text_col="text_clean", threshold=97):
    frame = frame.reset_index(drop=True)
    block = frame[text_col].str[:20]
    drop = set()
    for _, g in frame.groupby(block):
        if len(g) < 2:
            continue
        texts = g[text_col].tolist()
        sim = process.cdist(texts, texts, scorer=fuzz.ratio, workers=-1)
        gidx = g.index.to_numpy()
        for i in range(len(texts)):
            if gidx[i] in drop:
                continue
            dup = np.where(sim[i, i + 1:] >= threshold)[0] + i + 1
            drop.update(gidx[j] for j in dup)
    return frame.drop(index=list(drop)).reset_index(drop=True)

df = df_raw.dropna(subset=["text", "date"]).copy()
df["text"] = df["text"].astype(str).str.strip()
df = df[df["text"].str.len() >= 15].copy()
df["lang"] = df["text"].apply(detect_lang_safe)
df = df[df["lang"] == "en"].copy()
print("After language filter:", len(df))
df["text_clean"] = df["text"].apply(clean_text)
df = df[~df["text_clean"].apply(is_low_quality)]
df = df[~df["text_clean"].apply(is_spam_like)]
df = df.drop_duplicates(subset=["text_clean"]).reset_index(drop=True)
df = remove_near_duplicates(df, threshold=97)
df = df.reset_index(drop=True)
df["review_id"] = df.index
print("Final cleaned reviews:", len(df))

After language filter: 115735
Final cleaned reviews: 111233


## 4. Split reviews into short opinions


In [6]:
import spacy

nlp = spacy.load("en_core_web_sm",
                 disable=["tagger", "attribute_ruler", "lemmatizer", "ner"])
MIN_TOKENS = 7        # minimum words for a sentence on its own
MIN_PART_TOKENS = 4   # the two sides of a but/however split can be shorter
CONTRAST = re.compile(r"\b(?:but|however|although|though)\b", re.I)
_LEAD_CONNECT = re.compile(
    r"^(?:and|or|so|also|then|plus|because|but|however|which|that)\b[\s,]*", re.I)
# if a piece starts with is/are/have it lost its subject in the split
# ("are an extended family..."), so I throw it away. can't / doesn't are fine.
_SUBJECTLESS = re.compile(r"^(?:are|is|was|were|am|be|been|being|has|have|had)\b", re.I)
_DEPENDENT = re.compile(r"^(?:no matter|even if|even though|whether|as long as|in case)\b", re.I)

def _tidy(seg):
    seg = _LEAD_CONNECT.sub("", str(seg).strip(" ,.-"))
    return seg.strip(" ,.-")

def _valid(seg, min_tok):
    toks = seg.split()
    if len(toks) < min_tok:
        return False
    if _SUBJECTLESS.match(seg):
        return False
    if _DEPENDENT.match(seg) and len(toks) < 12:
        return False
    return True

def split_doc_into_segments(doc):
    segments = []
    for sent in doc.sents:
        s = _tidy(sent.text)
        if not s:
            continue
        parts = [_tidy(p) for p in CONTRAST.split(s)]
        parts = [p for p in parts if p]
        # only split on but/however if both sides stand on their own,
        # otherwise keep the sentence whole (the classifier is multi-label)
        if len(parts) >= 2 and all(_valid(p, MIN_PART_TOKENS) for p in parts):
            segments.extend(parts)
        elif _valid(s, MIN_TOKENS):
            segments.append(s)
    return segments

texts = df["text_clean"].astype(str).tolist()
segment_rows = []
for row, doc in zip(df.itertuples(index=False), nlp.pipe(texts, batch_size=256)):
    for seg_id, seg in enumerate(split_doc_into_segments(doc)):
        segment_rows.append({"review_id": row.review_id, "segment_id": seg_id,
                             "text_clean": seg, "rating": row.rating, "date": row.date})
segments_df = pd.DataFrame(segment_rows)
print("Total segments:", len(segments_df))

Total segments: 197939


## 5. Load the trained classifier (no training here)


In [7]:
from transformers import AutoTokenizer, AutoModelForSequenceClassification

_model_hits = glob.glob("/kaggle/input/**/netflix-swot-model", recursive=True) + \
              glob.glob("/kaggle/input/**/config.json", recursive=True)
MODEL_PATH = None
for h in _model_hits:
    d = h if os.path.isdir(h) else os.path.dirname(h)
    if os.path.exists(os.path.join(d, "config.json")):
        MODEL_PATH = d
        break
if MODEL_PATH is None:
    MODEL_PATH = "apolitidou/netflix-swot-classifier"   # HF Hub fallback

print("Loading classifier from:", MODEL_PATH)
classifier_tokenizer = AutoTokenizer.from_pretrained(MODEL_PATH)
classifier_model = AutoModelForSequenceClassification.from_pretrained(MODEL_PATH)
classifier_model.to("cuda" if torch.cuda.is_available() else "cpu").eval()
print("Classifier ready.")

@torch.inference_mode()
def predict_proba(texts, batch_size=256):
    device = classifier_model.device
    out = []
    for i in range(0, len(texts), batch_size):
        batch = [str(t) for t in texts[i:i + batch_size]]
        enc = classifier_tokenizer(batch, truncation=True, padding=True,
                                   max_length=96, return_tensors="pt").to(device)
        with torch.autocast("cuda", enabled=device.type == "cuda"):
            logits = classifier_model(**enc).logits
        out.append(torch.sigmoid(logits.float()).cpu().numpy())
    return np.vstack(out)

Loading classifier from: /kaggle/input/datasets/anastasiapolitidou/netflix-swot-model/netflix-swot-model


Loading weights:   0%|          | 0/393 [00:00<?, ?it/s]

Classifier ready.


## 6. Classify the fresh corpus


In [8]:
segments_reset = segments_df.reset_index(drop=True)
p_all = predict_proba(segments_reset["text_clean"].astype(str).tolist())
p_all = p_all * (prior_ratio ** ALPHA)
acc_mask = p_all >= thr_vec

frames = []
for c, lab in enumerate(SWOT_LABELS):
    idx = np.where(acc_mask[:, c])[0]
    f = segments_reset.iloc[idx].copy()
    f["swot_category"] = lab
    f["swot_confidence"] = p_all[idx, c]
    frames.append(f)
unc_idx = np.where(~acc_mask.any(axis=1))[0]
unc = segments_reset.iloc[unc_idx].copy()
unc["swot_category"] = UNCLASSIFIED
unc["swot_confidence"] = p_all[unc_idx].max(axis=1)
frames.append(unc)

df_swot = pd.concat(frames, ignore_index=True)
df_swot.to_csv("/kaggle/working/corpus_classified.csv", index=False)

CLASSIFIED = df_swot[df_swot["swot_category"] != UNCLASSIFIED].copy()
coverage = (df_swot["swot_category"] != UNCLASSIFIED).mean()
macro_f1 = MACRO_F1
print(f"Segment-label pairs: {len(df_swot)} | coverage: {coverage:.1%}")
print(df_swot["swot_category"].value_counts())

Segment-label pairs: 221168 | coverage: 99.7%
swot_category
Weakness        109514
Strength         50299
Threat           30320
Opportunity      30310
Unclassified       725
Name: count, dtype: int64


## 7. Embed each SWOT group

Every segment is turned into a vector with `all-mpnet-base-v2`. Themes are
induced and assigned in this same space in the next section.

In [9]:
from sentence_transformers import SentenceTransformer

# Only the embeddings are needed: themes are induced and assigned in this same
# vector space (section 8). Clustering is not used, so nothing here beyond the
# encoder is computed.
if torch.cuda.is_available():
    classifier_model.to("cpu")
gc.collect(); torch.cuda.empty_cache()

embedding_model = SentenceTransformer("all-mpnet-base-v2")


def embed_category(category):
    cat_df = CLASSIFIED[(CLASSIFIED["swot_category"] == category)
                        & CLASSIFIED["text_clean"].notna()].reset_index(drop=True)
    if len(cat_df) < 100:
        print(f"{category}: only {len(cat_df)} segments, skipped")
        return {"df": cat_df, "emb": None}
    emb = embedding_model.encode(cat_df["text_clean"].astype(str).tolist(),
                                 batch_size=256, show_progress_bar=True,
                                 normalize_embeddings=True).astype(np.float32)
    print(f"{category}: {len(cat_df):,} segments embedded")
    return {"df": cat_df, "emb": emb}


topic_data = {c: embed_category(c) for c in SWOT_LABELS}


modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

README.md: 0.00B [00:00, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/571 [00:00<?, ?B/s]

...

Weakness: 109,514 segments embedded


Batches:   0%|          | 0/119 [00:00<?, ?it/s]

Opportunity: 30,310 segments embedded


Batches:   0%|          | 0/119 [00:00<?, ?it/s]

Threat: 30,320 segments embedded


## 8. Themes

Clustering short review sentences produced either fragments or one huge blob,
so no label could describe them. This stage works the other way round:

1. A random sample of real reviews from the quadrant is read by the model,
   which lists the recurring themes it sees. Nothing is predefined.
2. Near-identical themes are merged.
3. **Every** segment in the corpus is then assigned to its closest theme by
   embedding similarity. This part is pure measurement.
4. A theme survives only if enough real reviews attach to it.

So the model proposes, the data decides. The counts, the ranking, the ratings
and the trends all come from the assignment, never from the model.

In [10]:
SPLIT_DATE = pd.to_datetime(CLASSIFIED["date"], errors="coerce").median()
CAT_RECENT_SHARE = {
    c: (pd.to_datetime(CLASSIFIED.loc[CLASSIFIED["swot_category"] == c, "date"],
                       errors="coerce") >= SPLIT_DATE).mean()
    for c in SWOT_LABELS}

def topic_trend(df_t, cat_share):
    d = pd.to_datetime(df_t["date"], errors="coerce").dropna()
    if len(d) < 10:
        return "STABLE"
    recent = (d >= SPLIT_DATE).mean()
    if recent >= cat_share + 0.10:
        return "RISING"
    if recent <= cat_share - 0.10:
        return "FALLING"
    return "STABLE"

def mmr_quotes(texts, emb, k=6, diversity=0.4, pool=50):
    centroid = emb.mean(axis=0)
    centroid = centroid / (np.linalg.norm(centroid) + 1e-9)
    sims = emb @ centroid
    cand = list(np.argsort(-sims)[:pool])
    selected = [cand[0]]
    while len(selected) < min(k * 2, len(cand)):
        rest = [c for c in cand if c not in selected]
        if not rest:
            break
        scores = [((1 - diversity) * sims[c]
                   - diversity * max(float(emb[c] @ emb[s]) for s in selected), c)
                  for c in rest]
        selected.append(max(scores)[1])
    picked = [texts[i].strip() for i in selected if len(texts[i].split()) >= 5][:k]
    return picked or [texts[cand[0]]]



from transformers import AutoModelForCausalLM, BitsAndBytesConfig

MIN_THEME_SHARE    = 0.005    # a theme must be raised by 0.5% of the quadrant
ASSIGN_MIN_SIM     = 0.30     # below this a segment belongs to no theme
MERGE_SIM          = 0.70     # two proposals this close are the same theme
SAMPLE_PER_CAT     = 320
BATCH              = 80

THEME_ID = "Qwen/Qwen2.5-7B-Instruct"
theme_tokenizer = AutoTokenizer.from_pretrained(THEME_ID)
if theme_tokenizer.pad_token is None:
    theme_tokenizer.pad_token = theme_tokenizer.eos_token
theme_model = AutoModelForCausalLM.from_pretrained(
    THEME_ID,
    quantization_config=BitsAndBytesConfig(
        load_in_4bit=True, bnb_4bit_compute_dtype=torch.float16,
        bnb_4bit_quant_type="nf4", bnb_4bit_use_double_quant=True),
    device_map="auto", max_memory={0: "14GiB", "cpu": "24GiB"},
    low_cpu_mem_usage=True)
theme_model.eval()

CATEGORY_BRIEF = {
    "Strength": "what users praise about the service",
    "Weakness": "what does not work or frustrates users",
    "Opportunity": "what users ask to be added or improved",
    "Threat": "what pushes users to cancel or leave for a competitor",
}


def propose_themes(reviews, category):
    """Asks the model which themes it sees in this sample. Free-form: no list
    of allowed themes is given, so the taxonomy comes from the reviews."""
    numbered = "\n".join(f"- {r}" for r in reviews)
    user_msg = f"""Below are real Netflix app reviews, all about {CATEGORY_BRIEF[category]}.

{numbered}

List the recurring themes you can see. For each theme write one line:
Theme Name | keyword, keyword, keyword

Rules:
- The theme name is 2 to 5 words, concrete, the way a business report would
  phrase it (for example: Frequent Price Increases, Subtitle Language Gaps).
- Only themes that several of these reviews share. Ignore one-off remarks.
- At most 8 themes. Return only the lines, nothing else."""
    messages = [{"role": "system",
                 "content": "You group customer feedback into recurring themes."},
                {"role": "user", "content": user_msg}]
    prompt = theme_tokenizer.apply_chat_template(messages, tokenize=False,
                                                 add_generation_prompt=True)
    enc = theme_tokenizer(prompt, return_tensors="pt", truncation=True,
                          max_length=3500).to(theme_model.device)
    with torch.no_grad():
        out = theme_model.generate(**enc, max_new_tokens=220, do_sample=False,
                                   pad_token_id=theme_tokenizer.pad_token_id)
    text = theme_tokenizer.decode(out[0][enc["input_ids"].shape[1]:],
                                  skip_special_tokens=True)
    themes = []
    for line in text.split("\n"):
        line = line.strip().lstrip("-*0123456789. ").strip()
        if "|" not in line:
            continue
        name, kw = line.split("|", 1)
        name = re.sub(r"[^A-Za-z0-9 &'-]", " ", name).strip()
        name = re.sub(r"\s+", " ", name)
        # the model sometimes echoes the name several times in one line
        _w = name.split()
        for _n in range(1, len(_w) // 2 + 1):
            if len(_w) % _n == 0 and all(_w[j] == _w[j % _n] for j in range(len(_w))):
                name = " ".join(_w[:_n])
                break
        kw = re.sub(r"[^A-Za-z0-9, '-]", " ", kw).strip()
        if 2 <= len(name.split()) <= 6 and kw:
            themes.append((name, kw))
    return themes


def merge_themes(themes):
    """Drops proposals that say the same thing as one already kept."""
    if not themes:
        return []
    texts = [f"{n}: {k}" for n, k in themes]
    emb = embedding_model.encode(texts, normalize_embeddings=True,
                                 show_progress_bar=False)
    kept, kept_emb = [], []
    for i, t in enumerate(themes):
        if kept_emb and max(float(emb[i] @ e) for e in kept_emb) >= MERGE_SIM:
            continue
        kept.append(t)
        kept_emb.append(emb[i])
    return kept


NEIGHBOUR_SIM = 0.55      # two segments this close say the same thing
DIVERSITY_SIM = 0.55      # one candidate per distinct opinion, so the
                          # sample mirrors how much each theme is discussed
REFERENCE_N   = 12000     # segments the density is measured against
CANDIDATE_N   = 30000


def popular_sample(cat_df, emb, k=None):
    """The segments the largest number of other reviews echo, one per distinct
    opinion. Deterministic: no random draw decides which themes are proposed."""
    k = k or SAMPLE_PER_CAT
    long_enough = cat_df["text_clean"].astype(str).str.split().str.len() >= 6
    idx = np.where(long_enough.to_numpy())[0]
    if len(idx) <= k:
        return cat_df["text_clean"].astype(str).iloc[idx].tolist()

    # evenly spaced, so the reference set covers the whole quadrant
    ref = idx[np.linspace(0, len(idx) - 1, min(REFERENCE_N, len(idx))).astype(int)]
    cand = idx[np.linspace(0, len(idx) - 1, min(CANDIDATE_N, len(idx))).astype(int)]
    ref_emb = emb[ref]

    density = np.empty(len(cand), dtype=np.int32)
    for s in range(0, len(cand), 4000):
        block = emb[cand[s:s + 4000]] @ ref_emb.T
        density[s:s + 4000] = (block >= NEIGHBOUR_SIM).sum(axis=1)

    texts = cat_df["text_clean"].astype(str).to_numpy()
    picked, picked_emb = [], []
    for j in np.argsort(-density):
        v = emb[cand[j]]
        if picked_emb and max(float(v @ p) for p in picked_emb) >= DIVERSITY_SIM:
            continue
        picked.append(texts[cand[j]])
        picked_emb.append(v)
        if len(picked) == k:
            break
    print(f"   density range of the selected opinions: "
          f"{density.max()} down to {int(np.sort(density)[-len(picked)])} echoes")
    return picked


def build_themes(category):
    bundle = topic_data[category]
    cat_df, emb = bundle["df"], bundle["emb"]
    if emb is None or len(cat_df) < 100:
        return pd.DataFrame()

    # 1. propose, from the most echoed opinions of the quadrant.
    #    A random sample gave a different taxonomy on every run and let rare
    #    opinions in. Instead each candidate is scored by how many OTHER
    #    segments of the quadrant say something close to it, and the densest
    #    ones are fed to the model. Those are, by construction, what most
    #    people are talking about, and the selection is deterministic.
    sample = popular_sample(cat_df, emb)
    proposals = []
    for i in range(0, len(sample), BATCH):
        proposals += propose_themes(sample[i:i + BATCH], category)
    themes = merge_themes(proposals)
    print(f"{category}: {len(proposals)} proposed -> {len(themes)} distinct themes")
    if not themes:
        return pd.DataFrame()

    # 2. assign every segment of the quadrant to its closest theme
    theme_emb = embedding_model.encode([f"{n}: {k}" for n, k in themes],
                                       normalize_embeddings=True,
                                       show_progress_bar=False).astype(np.float32)
    sims = emb @ theme_emb.T
    best = sims.argmax(axis=1)
    best_sim = sims.max(axis=1)
    assigned = best_sim >= ASSIGN_MIN_SIM
    cat_df = cat_df.copy()
    cat_df["theme_id"] = np.where(assigned, best, -1)
    print(f"{category}: {assigned.mean():.1%} of segments matched a theme")

    cat_share = CAT_RECENT_SHARE[category]
    min_mentions = max(50, int(MIN_THEME_SHARE * cat_df["review_id"].nunique()))
    rows = []
    centroids = {}
    for ti, (name, kw) in enumerate(themes):
        mask = (cat_df["theme_id"] == ti).to_numpy()
        if mask.sum() < 10:
            continue
        df_t = cat_df[mask]
        mentions = df_t["review_id"].nunique()
        if mentions < min_mentions:
            continue
        emb_t = emb[mask]
        texts_t = df_t["text_clean"].astype(str).tolist()
        quotes = mmr_quotes(texts_t, emb_t)
        centroid = emb_t.mean(axis=0)
        centroid = centroid / (np.linalg.norm(centroid) + 1e-9)
        centroids[name] = centroid
        # the segments nearest the centre of the theme, used to describe it
        central = np.argsort(-(emb_t @ centroid))[:8]
        row = {"topic_name": name,
               "central_texts": " || ".join(texts_t[i] for i in central),
               "topic_phrase": kw,
               "mention_count": mentions,
               "segment_count": int(mask.sum()),
               "coherence": round(float((emb_t @ centroid).mean()), 3),
               "avg_rating": round(df_t["rating"].mean(), 2),
               "avg_confidence": round(df_t["swot_confidence"].mean(), 3),
               "trend": topic_trend(df_t, cat_share)}
        for qi in range(6):
            row[f"quote_{qi + 1}"] = quotes[qi] if qi < len(quotes) else ""
        rows.append(row)
    table = pd.DataFrame(rows)
    if len(table):
        table = table.sort_values("mention_count", ascending=False).reset_index(drop=True)
        # two surviving themes whose centres are this close describe the same
        # thing; keep the one more reviews attached to
        keep, kept_c = [], []
        for idx, r in table.iterrows():
            c_ = centroids[r["topic_name"]]
            if kept_c and max(float(c_ @ k_) for k_ in kept_c) >= 0.80:
                continue
            keep.append(idx)
            kept_c.append(c_)
        table = table.loc[keep].reset_index(drop=True)
    return table, centroids


# ---- description: written from the theme's own reviews, checked semantically -
DESCRIBE_MIN_SIM = 0.45


def describe_theme(row, centroid):
    """One sentence written from the reviews closest to the centre of the
    theme. It is accepted only if its embedding still sits near that centre,
    so the check is semantic rather than word matching."""
    texts = [t for t in str(row["central_texts"]).split(" || ") if t.strip()][:8]
    if not texts:
        return ""
    evidence = "\n".join(f"- {t}" for t in texts)
    user_msg = f"""These reviews all belong to one theme: {row["topic_name"]}.

{evidence}

Write ONE sentence of 12 to 25 words describing what these users are saying.
Base it only on the reviews above. Return only the sentence."""
    messages = [{"role": "system",
                 "content": "You summarise what a group of customer reviews says."},
                {"role": "user", "content": user_msg}]
    prompt = theme_tokenizer.apply_chat_template(messages, tokenize=False,
                                                 add_generation_prompt=True)
    for sample in (False, True):
        torch.manual_seed(SEED)
        enc = theme_tokenizer(prompt, return_tensors="pt", truncation=True,
                              max_length=1600).to(theme_model.device)
        with torch.no_grad():
            out = theme_model.generate(
                **enc, max_new_tokens=60, do_sample=sample,
                temperature=0.7 if sample else None, top_p=0.9 if sample else None,
                pad_token_id=theme_tokenizer.pad_token_id)
        text = theme_tokenizer.decode(out[0][enc["input_ids"].shape[1]:],
                                      skip_special_tokens=True)
        text = text.strip().strip('"').split("\n")[0].strip()
        if not (8 <= len(text.split()) <= 35):
            continue
        if sum(ord(ch) > 591 for ch in text):
            continue
        v = embedding_model.encode([text], normalize_embeddings=True,
                                   show_progress_bar=False)[0]
        if float(v @ centroid) >= DESCRIBE_MIN_SIM:
            return text
    return ""


topic_tables, theme_centroids = {}, {}
for c in SWOT_LABELS:
    topic_tables[c], theme_centroids[c] = build_themes(c)

for c in SWOT_LABELS:
    tbl = topic_tables[c]
    if not len(tbl):
        continue
    tbl["business_insight"] = [
        describe_theme(r, theme_centroids[c][r["topic_name"]])
        for _, r in tbl.iterrows()]
    topic_tables[c] = tbl[tbl["business_insight"].str.len() > 0].reset_index(drop=True)

# theme_model is kept for the validation section; freed there.

for c in SWOT_LABELS:
    print(f"{c}: {len(topic_tables[c])} themes with a grounded description")


config.json:   0%|          | 0.00/663 [00:00<?, ?B/s]

tokenizer_config.json: 0.00B [00:00, ?B/s]

vocab.json: 0.00B [00:00, ?B/s]

merges.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

...

generation_config.json:   0%|          | 0.00/243 [00:00<?, ?B/s]

The following generation flags are not valid and may be ignored: ['temperature', 'top_p', 'top_k']. Set `TRANSFORMERS_VERBOSITY=info` for more details.


   density range of the selected opinions: 4094 down to 3222 echoes
Strength: 28 proposed -> 21 distinct themes
Strength: 80.9% of segments matched a theme
   density range of the selected opinions: 1647 down to 1127 echoes
Weakness: 32 proposed -> 21 distinct themes
Weakness: 54.5% of segments matched a theme
   density range of the selected opinions: 1523 down to 872 echoes
Opportunity: 32 proposed -> 26 distinct themes
Opportunity: 77.6% of segments matched a theme
   density range of the selected opinions: 3019 down to 2095 echoes
Threat: 33 proposed -> 24 distinct themes
Threat: 86.6% of segments matched a theme


The following generation flags are not valid and may be ignored: ['top_k']. Set `TRANSFORMERS_VERBOSITY=info` for more details.


Strength: 13 themes with a grounded description
Weakness: 17 themes with a grounded description
Opportunity: 12 themes with a grounded description
Threat: 10 themes with a grounded description


## 9. Final SWOT content

The five themes with the most reviews behind them, per quadrant.

In [11]:
TOP_N = 5

QUAD_TOTALS = {c: CLASSIFIED.loc[CLASSIFIED["swot_category"] == c, "review_id"].nunique()
               for c in SWOT_LABELS}

final_rows = []
for category in SWOT_LABELS:
    table = topic_tables[category]
    if not len(table):
        continue
    cand = table.head(TOP_N).copy()          # already ranked by reviews
    cand["swot"] = category
    cand["share"] = (cand["mention_count"] / QUAD_TOTALS[category] * 100).round(1)
    q_lo, q_hi = cand["share"].quantile([1 / 3, 2 / 3])
    cand["priority"] = np.select(
        [cand["share"] >= q_hi, cand["share"] >= q_lo], ["HIGH", "MEDIUM"], "LOW")
    final_rows.append(cand)

if not final_rows:
    raise RuntimeError("No theme survived in any quadrant - check MIN_THEME_SHARE.")
missing = [c for c in SWOT_LABELS if c not in {r["swot"].iloc[0] for r in final_rows}]
if missing:
    print(f"WARNING: no themes for {missing}; the matrix will be incomplete.")
final_swot = pd.concat(final_rows, ignore_index=True)
final_swot = final_swot[["swot", "priority", "trend", "topic_name", "business_insight",
                         "mention_count", "share", "avg_rating", "topic_phrase", "quote_1"]]
final_swot.to_csv("/kaggle/working/final_swot_matrix.csv", index=False)
print(final_swot[["swot", "priority", "topic_name", "business_insight",
                  "mention_count", "share"]].to_string())


           swot priority                   topic_name                                                                                                                        business_insight  mention_count  share
0      Strength     HIGH              Content Variety                                        Users praise Netflix for its extensive variety of movies and TV shows, including unique content.          10637   27.0
1      Strength     HIGH              User Experience                                                      Users find the app excellent with good functionality and enjoy using it regularly.           6329   16.0
2      Strength   MEDIUM               Mobile Viewing                                                                   Users highly recommend the app for watching both movies and TV shows.           1568    4.0
3      Strength      LOW         Device Compatibility                                          Users report the app functions well across various device

## 10. Evidence behind every theme

The check that decides whether the matrix is right is not another score: it is
reading the reviews that produced each theme. This section prints, for every
bullet in the SWOT matrix, its description, how many reviews are behind it, and
the five reviews closest to the centre of that theme. If those five read like
the theme, the theme holds.

Two numbers are printed next to each theme, both computed without any language
model: how similar the theme's own reviews are to it, and how similar the rest
of the quadrant is. The gap between them is what makes the theme a theme rather
than an arbitrary cut.

The full report is saved as `theme_evidence.txt`, so it can go straight into
the appendix of the thesis.

In [12]:
# =============================================================================
# Evidence report: every theme in the matrix, with the reviews behind it.
# This is the check that matters - read it and see whether each theme holds.
# Two mechanical numbers are added per theme; neither uses a language model.
# =============================================================================
report_lines, audit_rows = [], []

for cat in SWOT_LABELS:
    tbl = topic_tables[cat]
    if not len(tbl):
        continue
    th_emb = embedding_model.encode(
        [f"{n}: {k}" for n, k in zip(tbl["topic_name"], tbl["topic_phrase"])],
        normalize_embeddings=True, show_progress_bar=False).astype(np.float32)
    emb = topic_data[cat]["emb"]
    texts = topic_data[cat]["df"]["text_clean"].astype(str).tolist()
    sims = emb @ th_emb.T
    best, best_sim = sims.argmax(axis=1), sims.max(axis=1)
    keep = best_sim >= ASSIGN_MIN_SIM

    report_lines.append("=" * 78)
    report_lines.append(f"{cat.upper()}")
    report_lines.append("=" * 78)

    for ti in range(min(TOP_N, len(tbl))):
        row = tbl.iloc[ti]
        mine = keep & (best == ti)
        if mine.sum() == 0:
            continue
        # how close its own segments are, versus everything else in the quadrant
        own, other = float(sims[mine, ti].mean()), float(sims[~mine, ti].mean())
        # the reviews nearest the centre of the theme
        order = np.argsort(-np.where(mine, sims[:, ti], -1))[:5]

        report_lines.append("")
        report_lines.append(f"{ti + 1}. {row['topic_name']}   "
                            f"[{row['mention_count']:,} reviews | "
                            f"{row['avg_rating']}* | {row['trend']}]")
        report_lines.append(f"   {row['business_insight']}")
        report_lines.append(f"   separation: own {own:.2f} vs rest {other:.2f} "
                            f"(gap {own - other:+.2f})")
        report_lines.append("   reviews behind it:")
        for k in order:
            report_lines.append(f"     - {texts[k][:150]}")
        audit_rows.append({"swot": cat, "theme": row["topic_name"],
                           "reviews": int(row["mention_count"]),
                           "sim_own": round(own, 3), "sim_rest": round(other, 3),
                           "gap": round(own - other, 3)})
    report_lines.append("")

report = "\n".join(report_lines)
print(report)

with open("/kaggle/working/theme_evidence.txt", "w") as f:
    f.write(report)
audit = pd.DataFrame(audit_rows)
audit.to_csv("/kaggle/working/theme_evidence.csv", index=False)

print("=" * 78)
print(f"Every theme sits closer to its own reviews than to the rest of its "
      f"quadrant.\nSmallest gap in the matrix: {audit['gap'].min():+.2f} | "
      f"average gap: {audit['gap'].mean():+.2f}")
print("Saved theme_evidence.txt and theme_evidence.csv")

del theme_model
gc.collect(); torch.cuda.empty_cache()


STRENGTH

1. Content Variety   [10,637 reviews | 4.44* | STABLE]
   Users praise Netflix for its extensive variety of movies and TV shows, including unique content.
   separation: own 0.44 vs rest 0.19 (gap +0.24)
   reviews behind it:
     - variety of movies, shows, documentaries web series
     - the variety of content is incredible from blockbuster movies and trending series to documentaries and international shows
     - excellent content selection, from movies to top-notch original series
     - great variety of movies and tv shows
     - great variety of movies, tv shows series

2. User Experience   [6,329 reviews | 4.15* | STABLE]
   Users find the app excellent with good functionality and enjoy using it regularly.
   separation: own 0.43 vs rest 0.20 (gap +0.23)
   reviews behind it:
     - very less advertisement and even user friendly
     - smooth performance, user-friendly interface, and very helpful features
     - very good app doesn't have to many ads
     - amazing app

## 11. Export dashboard data (`data.json`)

In [13]:

import json, re
import numpy as np
import pandas as pd


def _py(o):
    """numpy/pandas scalars -> plain python, otherwise json.dump complains."""
    if isinstance(o, dict):          return {k: _py(v) for k, v in o.items()}
    if isinstance(o, (list, tuple)): return [_py(v) for v in o]
    if isinstance(o, np.integer):    return int(o)
    if isinstance(o, np.floating):   return float(o)
    if isinstance(o, np.bool_):      return bool(o)
    return o


_cl = CLASSIFIED.copy()
_cl["date"] = pd.to_datetime(_cl["date"], errors="coerce")
_cl["text_clean"] = _cl["text_clean"].astype(str)
_quote_lookup = {}
for _t, _r, _d in zip(_cl["text_clean"], _cl["rating"], _cl["date"]):
    _t = _t.strip()
    if _t and _t not in _quote_lookup:
        _quote_lookup[_t] = (int(_r) if pd.notna(_r) else 3,
                             _d.strftime("%b %Y") if pd.notna(_d) else "")


def _quote_meta(text, fallback_rating):
    text = str(text).strip()
    hit = _quote_lookup.get(text)
    if hit:
        return [hit[0], text, hit[1]]
    return [int(round(fallback_rating)), text, ""]


# topics: the final_swot rows, with the quotes taken back from topic_tables
_topics, _counters = [], {c: 0 for c in SWOT_LABELS}
_prefix = {"Strength": "s", "Weakness": "w", "Opportunity": "o", "Threat": "t"}
for _, row in final_swot.iterrows():
    cat = row["swot"]
    tbl = topic_tables.get(cat, pd.DataFrame())
    match = tbl[tbl["topic_name"] == row["topic_name"]] if len(tbl) else tbl
    if len(match):
        raw_quotes = [match.iloc[0].get(f"quote_{i}", "") for i in range(1, 7)]
    else:
        raw_quotes = [row.get("quote_1", "")]
    avg_r = float(row["avg_rating"])
    quotes = [_quote_meta(q, avg_r) for q in raw_quotes if str(q).strip()]
    _counters[cat] += 1
    _topics.append({
        "id": f"{_prefix[cat]}{_counters[cat]}",
        "quad": cat,
        "name": str(row["topic_name"]),
        "phrase": str(row["topic_phrase"]),
        "mentions": int(row["mention_count"]),
        "avgRating": round(avg_r, 1),
        "trend": str(row["trend"]),
        "prio": str(row["priority"]),
        "insight": str(row["business_insight"]),
        "quotes": quotes,
    })

_meta = {
    "total_reviews":      int(df_swot["review_id"].nunique()),
    "total_opinions":     int(len(df_swot)),
    "avg_rating":         round(float(df_raw["rating"].mean()), 2),
    "positive_share_pct": round(float((df_raw["rating"] >= 4).mean()) * 100, 1),
    "coverage_pct":       round(float(globals().get("coverage",
                              (df_swot["swot_category"] != UNCLASSIFIED).mean())) * 100, 1),
    "macro_f1":           round(float(globals().get("macro_f1", 0.0)), 2),
}

_rr = df_raw.dropna(subset=["date"]).copy()
_rr["date"] = pd.to_datetime(_rr["date"], errors="coerce")
_rr = _rr.dropna(subset=["date"]).sort_values("date")
_rr["month"] = _rr["date"].dt.to_period("M")
_this_month = pd.Timestamp.today().to_period("M")
_months = [m for m in sorted(_rr["month"].unique()) if m < _this_month]
_history = []
for _m in (_months[-6:] if len(_months) >= 6 else _months):
    _upto = _rr[_rr["month"] <= _m]
    _history.append({
        "total_reviews":      int(len(_upto)),
        "avg_rating":         round(float(_upto["rating"].mean()), 2),
        "positive_share_pct": round(float((_upto["rating"] >= 4).mean()) * 100, 1),
    })

_quadTotals = {c: int(QUAD_TOTALS.get(c, 0)) for c in SWOT_LABELS}


_ct = pd.crosstab(CLASSIFIED["rating"], CLASSIFIED["swot_category"]) \
        .reindex(index=[1, 2, 3, 4, 5], columns=SWOT_LABELS).fillna(0)
_ct_pct = _ct.div(_ct.sum(axis=1).replace(0, np.nan), axis=0).fillna(0) * 100
_heatmap = {c: [int(round(_ct_pct.loc[r, c])) for r in [1, 2, 3, 4, 5]] for c in SWOT_LABELS}


_rc = df_raw["rating"].value_counts().reindex([1, 2, 3, 4, 5]).fillna(0)
_ratingCounts = [int(_rc.loc[r]) for r in [1, 2, 3, 4, 5]]


_cq = CLASSIFIED.copy()
_cq["date"] = pd.to_datetime(_cq["date"], errors="coerce")
_cq = _cq.dropna(subset=["date"])
_cq["q"] = _cq["date"].dt.to_period("Q")
_qidx = [q for q in sorted(_cq["q"].unique()) if q >= pd.Period("2021Q1")]
_tct = pd.crosstab(_cq["q"], _cq["swot_category"]) \
         .reindex(index=_qidx, columns=SWOT_LABELS).fillna(0)
_tct_pct = _tct.div(_tct.sum(axis=1).replace(0, np.nan), axis=0).fillna(0) * 100
_trend = {c: [round(float(_tct_pct.loc[q, c]), 1) for q in _qidx] for c in SWOT_LABELS}


_vidx = [q for q in sorted(_cq["q"].unique()) if q >= pd.Period("2020Q1")]
_vc = _cq["q"].value_counts().reindex(_vidx).fillna(0)
_volume_counts = [round(float(_vc.loc[q]) / 1000, 1) for q in _vidx]
_lo = _vidx.index(pd.Period("2020Q2")) if pd.Period("2020Q2") in _vidx else 1
_hi = _vidx.index(pd.Period("2021Q2")) if pd.Period("2021Q2") in _vidx else 5

_max_d = pd.to_datetime(df_raw["date"], errors="coerce").max()
_period_label = "Jan 2020 \u2013 " + (_max_d.strftime("%b %Y") if pd.notna(_max_d) else "now")

data_json = _py({
    "generated_at": pd.Timestamp.now(tz="UTC").strftime("%Y-%m-%d %H:%M UTC"),
    "period_label": _period_label,
    "meta":         _meta,
    "history":      _history,
    "quadTotals":   _quadTotals,
    "topics":       _topics,
    "heatmap":      _heatmap,
    "ratingCounts": _ratingCounts,
    "trend":        _trend,
    "volume":       {"counts": _volume_counts, "covid_range": [_lo, _hi]},
})

_OUT = "/kaggle/working/data.json"
with open(_OUT, "w", encoding="utf-8") as _f:
    json.dump(data_json, _f, ensure_ascii=False, indent=2)

print("Wrote", _OUT)
print(json.dumps(data_json["meta"], indent=2))
print(f"topics: {len(data_json['topics'])} | "
      f"trend quarters: {len(_trend['Strength'])} | "
      f"volume quarters: {len(_volume_counts)}")


Wrote /kaggle/working/data.json
{
  "total_reviews": 102552,
  "total_opinions": 221168,
  "avg_rating": 2.85,
  "positive_share_pct": 42.3,
  "coverage_pct": 99.7,
  "macro_f1": 0.8
}
topics: 20 | trend quarters: 23 | volume quarters: 27
